# BERTScore Evaluation Notebook

This notebook evaluates the three submitted model runs against `solutions.csv` and writes the aggregate and per-example BERTScore outputs into the `evaluation` folder.


In [ ]:
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from statistics import fmean

import bert_score
import evaluate
import pandas as pd
from IPython.display import display


REFERENCE_COLUMN_CANDIDATES = ("answer", "solution", "reference", "correct_answer")
PREDICTION_COLUMN_CANDIDATES = ("answer", "prediction", "predicted_answer")
DEFAULT_MODEL_TYPE = "bert-base-multilingual-cased"


def resolve_evaluation_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "evaluation", cwd.parent / "evaluation"]
    for candidate in candidates:
        if (candidate / "solutions.csv").exists() and (candidate.parent / "results").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the evaluation folder. Run this notebook from the evaluation directory or the submission root."
    )


EVALUATION_DIR = resolve_evaluation_dir()
SUBMISSION_ROOT = EVALUATION_DIR.parent
RESULTS_DIR = SUBMISSION_ROOT / "results"

REFERENCE_PATH = EVALUATION_DIR / "solutions.csv"
SUMMARY_PATH = EVALUATION_DIR / "bertscore_summary.csv"
DETAILS_PATH = EVALUATION_DIR / "bertscore_details.csv"
MODEL_TYPE = DEFAULT_MODEL_TYPE
WRITE_DETAILS = True

print(f"Evaluation directory: {EVALUATION_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Reference file: {REFERENCE_PATH}")


In [ ]:
@dataclass(frozen=True)
class ModelRun:
    label: str
    path: Path


def load_rows(path: Path) -> tuple[list[dict[str, str]], list[str]]:
    if not path.exists():
        raise FileNotFoundError(f"Missing CSV file: {path}")

    separators_to_try = [";", ","]
    frame = None
    for separator in separators_to_try:
        try:
            candidate = pd.read_csv(
                path,
                sep=separator,
                dtype=str,
                encoding="utf-8-sig",
                keep_default_na=False,
                engine="python",
            )
        except Exception:
            continue
        if len(candidate.columns) > 1:
            frame = candidate
            break
    else:
        raise ValueError(f"Could not parse a usable header from {path}")

    assert frame is not None
    frame.columns = [str(column).strip() for column in frame.columns]
    rows = []
    for row in frame.to_dict(orient="records"):
        rows.append({str(key).strip(): str(value or "").strip() for key, value in row.items()})
    return rows, list(frame.columns)


def choose_column(fieldnames: list[str], candidates: tuple[str, ...], *, file_label: str) -> str:
    lowered = {name.lower(): name for name in fieldnames}
    for candidate in candidates:
        if candidate in lowered:
            return lowered[candidate]
    raise ValueError(
        f"{file_label} must contain one of these columns: {', '.join(candidates)}. "
        f"Found columns: {', '.join(fieldnames)}"
    )


def validate_and_index(
    rows: list[dict[str, str]],
    *,
    path: Path,
    text_column: str,
    allow_duplicate_ids: bool = False,
) -> tuple[dict[str, str], list[str]]:
    indexed: dict[str, str] = {}
    duplicate_ids: list[str] = []
    for row_number, row in enumerate(rows, start=2):
        if "id" not in row:
            raise ValueError(f"{path} is missing the required 'id' column.")
        row_id = row["id"].strip()
        if not row_id:
            raise ValueError(f"{path} has an empty id at CSV row {row_number}.")
        if row_id in indexed:
            if allow_duplicate_ids:
                duplicate_ids.append(row_id)
                continue
            raise ValueError(f"{path} contains a duplicate id: {row_id}")
        if text_column not in row:
            raise ValueError(f"{path} is missing the required text column '{text_column}'.")
        indexed[row_id] = row[text_column].strip()
    return indexed, duplicate_ids


def build_model_runs(results_dir: Path) -> list[ModelRun]:
    return [
        ModelRun("Inference_Qwen3", results_dir / "inference_qwen3_0_6b.csv"),
        ModelRun("FineTune_Qwen3", results_dir / "finetune_qwen3_0_6b.csv"),
        ModelRun("RAG_Qwen3", results_dir / "rag_qwen3_0_6b.csv"),
    ]


def resolve_local_model_path(model_type: str) -> str:
    if model_type != DEFAULT_MODEL_TYPE:
        return model_type

    snapshot_root = Path.home() / ".cache" / "huggingface" / "hub" / "models--bert-base-multilingual-cased" / "snapshots"
    if not snapshot_root.exists():
        return model_type

    snapshots = sorted(child for child in snapshot_root.iterdir() if child.is_dir())
    if not snapshots:
        return model_type

    return snapshots[-1].as_posix()


def compute_bertscore(*, references: list[str], predictions: list[str], model_type: str) -> dict[str, list[float]]:
    metric = evaluate.load("bertscore")
    resolved_model_type = resolve_local_model_path(model_type)
    num_layers = None
    if resolved_model_type != model_type and model_type in bert_score.utils.model2layers:
        num_layers = bert_score.utils.model2layers[model_type]
    return metric.compute(
        predictions=predictions,
        references=references,
        model_type=resolved_model_type,
        num_layers=num_layers,
        batch_size=8,
        verbose=False,
        use_fast_tokenizer=False,
    )


def write_frame(path: Path, rows: list[dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(path, index=False, encoding="utf-8")


In [ ]:
reference_rows, reference_fields = load_rows(REFERENCE_PATH)
reference_column = choose_column(reference_fields, REFERENCE_COLUMN_CANDIDATES, file_label=str(REFERENCE_PATH))
reference_by_id, duplicate_reference_ids = validate_and_index(
    reference_rows,
    path=REFERENCE_PATH,
    text_column=reference_column,
    allow_duplicate_ids=True,
)

if duplicate_reference_ids:
    duplicate_counter = Counter(duplicate_reference_ids)
    duplicate_preview = ", ".join(f"{row_id} x{count + 1}" for row_id, count in duplicate_counter.most_common(5))
    print(
        f"Warning: {REFERENCE_PATH.name} contains duplicate ids. "
        f"Keeping the first occurrence for evaluation: {duplicate_preview}"
    )

summary_rows: list[dict[str, object]] = []
detail_rows: list[dict[str, object]] = []

for run in build_model_runs(RESULTS_DIR):
    prediction_rows, prediction_fields = load_rows(run.path)
    prediction_column = choose_column(prediction_fields, PREDICTION_COLUMN_CANDIDATES, file_label=str(run.path))
    prediction_by_id, _ = validate_and_index(prediction_rows, path=run.path, text_column=prediction_column)

    reference_ids = set(reference_by_id)
    prediction_ids = set(prediction_by_id)
    shared_ids = [row_id for row_id in prediction_by_id if row_id in reference_by_id]
    missing_reference_ids = sorted(prediction_ids - reference_ids)
    extra_reference_ids = sorted(reference_ids - prediction_ids)

    if not shared_ids:
        raise ValueError(f"No overlapping ids were found between {REFERENCE_PATH.name} and {run.path.name}.")

    if missing_reference_ids or extra_reference_ids:
        print(
            f"Warning: using the shared evaluation subset for {run.label}. "
            f"Matched ids: {len(shared_ids)} | "
            f"Missing references for prediction ids: {missing_reference_ids[:5]} | "
            f"Reference-only ids: {extra_reference_ids[:5]}"
        )

    ordered_references = [reference_by_id[row_id] for row_id in shared_ids]
    ordered_predictions = [prediction_by_id[row_id] for row_id in shared_ids]
    bertscore = compute_bertscore(
        references=ordered_references,
        predictions=ordered_predictions,
        model_type=MODEL_TYPE,
    )

    summary_rows.append(
        {
            "model": run.label,
            "num_examples": len(shared_ids),
            "bertscore_precision": f"{fmean(bertscore['precision']):.6f}",
            "bertscore_recall": f"{fmean(bertscore['recall']):.6f}",
            "bertscore_f1": f"{fmean(bertscore['f1']):.6f}",
        }
    )

    if WRITE_DETAILS:
        for row_id, reference, prediction, precision, recall, f1 in zip(
            shared_ids,
            ordered_references,
            ordered_predictions,
            bertscore["precision"],
            bertscore["recall"],
            bertscore["f1"],
            strict=True,
        ):
            detail_rows.append(
                {
                    "model": run.label,
                    "id": row_id,
                    "reference": reference,
                    "prediction": prediction,
                    "bertscore_precision": f"{precision:.6f}",
                    "bertscore_recall": f"{recall:.6f}",
                    "bertscore_f1": f"{f1:.6f}",
                }
            )

write_frame(SUMMARY_PATH, summary_rows)
if WRITE_DETAILS:
    write_frame(DETAILS_PATH, detail_rows)

summary_frame = pd.DataFrame(summary_rows)
display(summary_frame)
print(f"Wrote summary to {SUMMARY_PATH}")
if WRITE_DETAILS:
    print(f"Wrote details to {DETAILS_PATH}")
